# Classificação e NER para Ouvidoria Municipal

Notebook simplificado para teste local sem necessidade de modelos externos.

In [ ]:
import os
import re
import json
import pandas as pd

## Configuração

In [ ]:
CATEGORIES = ["Infraestrutura", "Saúde", "Trânsito", "Iluminação", "Outros"]

ROUTING_MAP = {
    "Infraestrutura": "Secretaria de Obras e Infraestrutura",
    "Saúde": "Secretaria Municipal de Saúde",
    "Trânsito": "DETRAN / Secretaria de Mobilidade",
    "Iluminação": "Secretaria de Serviços Urbanos",
    "Outros": "Ouvidoria Geral"
}

print("Configuração carregada")

## Dataset de Teste

In [ ]:
dataset = [
    "Tem um buraco enorme na Av. Principal, perto do posto de saúde, já faz 3 dias",
    "O semáforo da esquina da Rua 5 está quebrado há uma semana",
    "Falta dipirona no posto de saúde do bairro Centro",
    "O poste da Rua das Flores está sem luz há 3 noites",
    "A calçada da escola municipal está toda quebrada"
]

df = pd.DataFrame({
    "texto": dataset,
    "categoria_esperada": ["Infraestrutura", "Trânsito", "Saúde", "Iluminação", "Infraestrutura"]
})

print(f"Dataset: {len(df)} exemplos")
df

## Classificação por Palavras-Chave

In [ ]:
def classify_by_keywords(text):
    text_lower = text.lower()
    
    keywords = {
        "Infraestrutura": ["buraco", "rua", "avenida", "calçada", "ponte", "água", "bomba", "bueiro"],
        "Saúde": ["posto", "saúde", "médico", "hospital", "enfermeira", "farmácia", "medicamento"],
        "Trânsito": ["ônibus", "semáforo", "trânsito", "placa", "transporte", "radar"],
        "Iluminação": ["luz", "poste", "iluminação", "lâmpada", "lampada", "escuro"]
    }
    
    scores = {cat: sum(1 for w in words if w in text_lower) for cat, words in keywords.items()}
    
    if max(scores.values()) > 0:
        categoria = max(scores, key=scores.get)
        confianca = min(scores[categoria] / 3, 0.95)
        return {"categoria": categoria, "confianca": confianca, "source": "keyword"}
    
    return {"categoria": "Outros", "confianca": 0.5, "source": "default"}

print("Classificação pronta")

## Extração de Entidades (NER)

In [ ]:
def extract_entities(text):
    result = {"localizacao": [], "organizacao": [], "nome_servidor": [], "equipamento": [], "data": [], "urgencia": "media"}
    lower = text.lower()
    
    # Localização
    locais = re.findall(r'\b(rua|avenida|av\.|praça)\s+[\w\s]+', text, re.I)
    result["localizacao"] = [l.strip().title() for l in locais]
    
    # Organização
    orgs = re.findall(r'(?:posto|ubs|hospital|escola)\s+[\w\s]+', text, re.I)
    result["organizacao"] = [o.strip() for o in orgs]
    
    # Equipamento
    equipamentos = {"Semáforo": ["semáforo"], "Poste": ["poste", "luz"], "Calçada": ["calçada"]}
    for eq, kws in equipamentos.items():
        if any(k in lower for k in kws):
            result["equipamento"].append(eq)
    
    # Data
    datas = re.findall(r'\d+\s*(?:dias?|semanas?|noites?)', lower)
    result["data"] = datas
    
    # Urgência
    if any(w in lower for w in ["urgente", "emergência", "risco", "perigoso"]):
        result["urgencia"] = "alta"
    elif any(w in lower for w in ["semana", "mês", "antigo"]):
        result["urgencia"] = "baixa"
    
    return result

print("NER pronto")

## Pipeline Completo

In [ ]:
def processar(texto):
    resultado_class = classify_by_keywords(texto)
    entidades = extract_entities(texto)
    return {
        "texto": texto,
        "categoria": resultado_class["categoria"],
        "confianca": resultado_class["confianca"],
        "secretaria": ROUTING_MAP.get(resultado_class["categoria"]),
        "entidades": entidades
    }

print("Pipeline pronto")

## Testes

In [ ]:
for texto in dataset:
    r = processar(texto)
    print(f"Texto: {texto[:50]}...")
    print(f"  Categoria: {r['categoria']} ({r['confianca']:.2f})")
    print(f"  Secretaria: {r['secretaria']}")
    print(f"  Urgência: {r['entidades']['urgencia']}")
    print()

## Avaliação

In [ ]:
resultados = []
acertos = 0

for idx, row in df.iterrows():
    r = processar(row["texto"])
    acerto = row["categoria_esperada"] == r["categoria"]
    acertos += acerto
    resultados.append({"esperada": row["categoria_esperada"], "predita": r["categoria"], "acerto": acerto})

accuracy = acertos / len(df)
print(f"Accuracy: {accuracy * 100:.2f}%")
pd.DataFrame(resultados)

## Conclusão

In [ ]:
print("Notebook concluído com sucesso!")
print("Este notebook usa apenas classificação por palavras-chave e regex para NER.")
print("Para produção, considere integrar com modelos BERTimbau fine-tuned ou Cohere API.")